In [0]:

# Traffic data location
traffic_path = "/Volumes/workspace/default/urbanpulse_bronze/traffic"

# List uploaded folders and files
traffic_items = dbutils.fs.ls(traffic_path)

for item in traffic_items:
    print(item.name, item.size)


In [0]:

probe_path = traffic_path + "/probe_counts/geojson"

probe_files = dbutils.fs.ls(probe_path)

print("Number of files:", len(probe_files))

for file in probe_files:
    print(file.name, file.size)


In [0]:

# Select one daily file
first_file = probe_files[0].path

print("Inspecting:", first_file)

# Read one file
traffic_sample = spark.read.option(
    "multiline", "true"
).json(first_file)

print("Schema:")
traffic_sample.printSchema()

display(traffic_sample.limit(5))


In [0]:
traffic_path = "/Volumes/workspace/default/urbanpulse_bronze/traffic"

probe_path = traffic_path + "/probe_counts/geojson"

probe_files = dbutils.fs.ls(probe_path)

print("Total probe files:", len(probe_files))

for f in probe_files:
    print(f.name, f.size)

In [0]:
first_file = probe_files[0].path

traffic_sample = spark.read.option(
    "multiline", "true"
).json(first_file)

traffic_sample.printSchema()

display(traffic_sample.limit(5))

In [0]:

from pyspark.sql import functions as F

# Convert the features array into individual road segments
traffic_features = traffic_sample.select(
    F.explode("features").alias("feature")
)

# Inspect important properties of the first five segments
display(
    traffic_features.select(
        F.col("feature.properties.segmentId").alias("segment_id"),
        F.col("feature.properties.streetName").alias("street_name"),
        F.col("feature.properties.frc").alias("frc"),
        F.col("feature.properties.distance").alias("distance"),
        F.col("feature.properties.speedLimit").alias("speed_limit"),
        F.col("feature.properties.segmentProbeCounts").alias("probe_counts"),
        F.col("feature.properties.dateRanges").alias("date_ranges"),
        F.col("feature.properties.timeSets").alias("time_sets")
    ).limit(5)
)


In [0]:

# Count road segments in the sample file
print("Road segments:", traffic_features.count())

# Calculate the total number of probe measurements
probe_summary = traffic_features.select(
    F.explode_outer(
        "feature.properties.segmentProbeCounts"
    ).alias("probe")
)

print("Probe measurements:", probe_summary.count())

display(probe_summary.limit(10))


In [0]:

from pyspark.sql import functions as F

# Identify actual road segments
traffic_segments = traffic_features.filter(
    F.col("feature.properties.segmentId").isNotNull()
)

# Identify metadata records
traffic_metadata = traffic_features.filter(
    F.col("feature.properties.dateRanges").isNotNull()
)

print("Actual road segments:", traffic_segments.count())
print("Metadata records:", traffic_metadata.count())

# Inspect date-range metadata
display(
    traffic_metadata
    .select(F.explode("feature.properties.dateRanges").alias("date_range"))
    .select(
        F.col("date_range.@id").alias("date_range_id"),
        F.col("date_range.from").alias("start_date"),
        F.col("date_range.to").alias("end_date")
    )
)

# Inspect hourly time-set metadata
display(
    traffic_metadata
    .select(F.explode("feature.properties.timeSets").alias("time_set"))
    .select(
        F.col("time_set.@id").alias("time_set_id"),
        F.col("time_set.name").alias("time_interval")
    )
    .orderBy("time_set_id")
)


In [0]:
from pyspark.sql import functions as F

# Extract date-range metadata
date_ranges = (
    traffic_metadata
    .select(F.explode("feature.properties.dateRanges").alias("date_range"))
    .select(
        F.col("date_range.@id").alias("date_range_id"),
        F.col("date_range.from").alias("start_date"),
        F.col("date_range.to").alias("end_date")
    )
)

display(date_ranges)

# Extract hourly time-set metadata
time_sets = (
    traffic_metadata
    .select(F.explode("feature.properties.timeSets").alias("time_set"))
    .select(
        F.col("time_set.@id").alias("time_set_id"),
        F.col("time_set.name").alias("time_interval")
    )
    .orderBy("time_set_id")
)

display(time_sets)

In [0]:
from pyspark.sql import functions as F

# Extract date mappings
date_ranges = (
    traffic_metadata
    .select(F.explode("feature.properties.dateRanges").alias("d"))
    .select(
        F.col("d.@id").alias("date_range_id"),
        F.to_date("d.from").alias("traffic_date"),
        F.to_date("d.to").alias("end_date")
    )
)

# Extract hourly mappings
time_sets = (
    traffic_metadata
    .select(F.explode("feature.properties.timeSets").alias("t"))
    .select(
        F.col("t.@id").alias("time_set_id"),
        F.col("t.name").alias("time_interval")
    )
)

print("DATE MAPPING")
display(date_ranges.orderBy("date_range_id"))

print("TIME MAPPING")
display(time_sets.orderBy("time_set_id"))

In [0]:
from pyspark.sql import functions as F

probe_summary = (
    traffic_segments
    .select(
        F.col("feature.properties.segmentId").alias("segment_id"),
        F.col("feature.properties.streetName").alias("street_name"),
        F.col("feature.properties.frc").alias("frc"),
        F.col("feature.properties.distance").alias("distance"),
        F.col("feature.properties.speedLimit").alias("speed_limit"),
        F.col("feature.geometry.coordinates").alias("coordinates"),
        F.explode("feature.properties.segmentProbeCounts").alias("probe")
    )
    .select(
        "segment_id",
        "street_name",
        "frc",
        "distance",
        "speed_limit",
        "coordinates",
        F.col("probe.dateRange").alias("date_range_id"),
        F.col("probe.timeSet").alias("time_set_id"),
        F.col("probe.probeCount").alias("probe_count")
    )
)

print("Flattened measurement rows:", probe_summary.count())
display(probe_summary.limit(10))


In [0]:
traffic_enriched = (
    probe_summary
    .join(date_ranges, on="date_range_id", how="left")
    .join(time_sets, on="time_set_id", how="left")
)

display(traffic_enriched.limit(10))

In [0]:
traffic_enriched.select(
    F.count("*").alias("total_rows"),
    F.count("traffic_date").alias("valid_dates"),
    F.count("time_interval").alias("valid_time_intervals"),
    F.sum(
        F.when(F.col("probe_count").isNull(), 1).otherwise(0)
    ).alias("null_probe_counts")
).show()

In [0]:
traffic_enriched.select(
    "segment_id",
    "street_name",
    "traffic_date",
    "time_interval",
    "probe_count"
).show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

traffic_clean = (
    traffic_enriched
    .filter(F.col("segment_id").isNotNull())
    .filter(F.col("traffic_date").isNotNull())
    .filter(F.col("time_interval").isNotNull())
    .filter(F.col("probe_count").isNotNull())
    .filter(F.col("probe_count") >= 0)
    .dropDuplicates([
        "segment_id",
        "traffic_date",
        "time_interval"
    ])
)

In [0]:
traffic_clean = (
    traffic_clean
    .withColumn(
        "traffic_hour",
        F.split(F.col("time_interval"), "-")[0]
    )
    .withColumn(
        "traffic_hour",
        F.split(F.col("traffic_hour"), ":")[0].cast("int")
    )
)

In [0]:
print("Original rows:", traffic_enriched.count())
print("Cleaned rows:", traffic_clean.count())

traffic_clean.printSchema()

display(
    traffic_clean.select(
        "segment_id",
        "street_name",
        "traffic_date",
        "traffic_hour",
        "probe_count"
    ).limit(10)
)

In [0]:
from pyspark.sql import functions as F

traffic_clean.select(
    F.count("*").alias("total_records"),
    F.countDistinct("segment_id").alias("unique_road_segments"),
    F.countDistinct("traffic_date").alias("unique_dates"),
    F.sum(F.when(F.col("probe_count") < 0, 1).otherwise(0)).alias("negative_counts"),
    F.sum(F.when(F.col("traffic_hour").isNull(), 1).otherwise(0)).alias("invalid_hours")
).show()

In [0]:
silver_table = "workspace.default.silver_traffic"

(
    traffic_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("Traffic Silver table saved successfully!")

In [0]:
silver_traffic_df = spark.table("workspace.default.silver_traffic")

print("Saved records:", silver_traffic_df.count())
display(silver_traffic_df.limit(10))

In [0]:
silver_table = "workspace.default.silver_traffic"

(
    traffic_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("First daily traffic file saved successfully!")


In [0]:
silver_traffic_df = spark.table("workspace.default.silver_traffic")

print("Saved records:", silver_traffic_df.count())
print("Unique road segments:", silver_traffic_df.select("segment_id").distinct().count())

display(silver_traffic_df.limit(10))

In [0]:
probe_path = "/Volumes/workspace/default/urbanpulse_bronze/traffic/probe_counts/geojson"

probe_files = dbutils.fs.ls(probe_path)

daily_files = [
    f for f in probe_files
    if f.name.lower().endswith((".json", ".geojson"))
]

print("Total daily GeoJSON files:", len(daily_files))

for file in sorted(daily_files, key=lambda x: x.name):
    print(file.name, "|", round(file.size / (1024 * 1024), 2), "MB")

In [0]:
second_file = next(
    f.path for f in daily_files
    if "2024-08-12" in f.name
)

traffic_day2 = (
    spark.read
    .option("multiline", "true")
    .json(second_file)
)

traffic_day2.printSchema()

print("Total features:", traffic_day2.count())

display(traffic_day2.limit(3))


In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Paths and target table
probe_path = "/Volumes/workspace/default/urbanpulse_bronze/traffic/probe_counts/geojson"
target_table = "workspace.default.silver_traffic"
staging_table = "workspace.default.traffic_staging"

# Discover all daily files except the first day already processed
all_files = [
    f for f in dbutils.fs.ls(probe_path)
    if f.name.lower().endswith((".json", ".geojson"))
]

remaining_files = sorted(
    [f for f in all_files if "2024-08-11" not in f.name],
    key=lambda x: x.name
)

print("Files to process:", len(remaining_files))

# Process one file at a time
for index, file in enumerate(remaining_files, start=1):

    print(f"\nProcessing {index}/{len(remaining_files)}: {file.name}")

    raw_df = (
        spark.read
        .option("multiline", "true")
        .json(file.path)
    )

    # Separate road segments and metadata
    features = raw_df.select(
        F.explode("features").alias("feature")
    )

    metadata = features.filter(
        F.col("feature.properties.segmentId").isNull()
        & F.col("feature.properties.dateRanges").isNotNull()
    )

    segments = features.filter(
        F.col("feature.properties.segmentId").isNotNull()
    )

    # Extract date mapping
    date_ranges = (
        metadata
        .select(F.explode("feature.properties.dateRanges").alias("d"))
        .select(
            F.col("d.@id").alias("date_range_id"),
            F.to_date("d.from").alias("traffic_date"),
            F.to_date("d.to").alias("end_date")
        )
    )

    # Extract hourly mapping
    time_sets = (
        metadata
        .select(F.explode("feature.properties.timeSets").alias("t"))
        .select(
            F.col("t.@id").alias("time_set_id"),
            F.col("t.name").alias("time_interval")
        )
    )

    # Flatten probe measurements
    daily_df = (
        segments
        .select(
            F.col("feature.properties.segmentId").alias("segment_id"),
            F.col("feature.properties.streetName").alias("street_name"),
            F.col("feature.properties.frc").alias("frc"),
            F.col("feature.properties.distance").alias("distance"),
            F.col("feature.properties.speedLimit").alias("speed_limit"),
            F.col("feature.geometry.coordinates").alias("coordinates"),
            F.explode(
                "feature.properties.segmentProbeCounts"
            ).alias("probe")
        )
        .select(
            "segment_id",
            "street_name",
            "frc",
            "distance",
            "speed_limit",
            "coordinates",
            F.col("probe.dateRange").alias("date_range_id"),
            F.col("probe.timeSet").alias("time_set_id"),
            F.col("probe.probeCount").alias("probe_count")
        )
        .join(date_ranges, "date_range_id", "left")
        .join(time_sets, "time_set_id", "left")
        .filter(
            F.col("traffic_date").isNotNull()
            & F.col("time_interval").isNotNull()
            & F.col("probe_count").isNotNull()
            & (F.col("probe_count") >= 0)
        )
        .withColumn(
            "traffic_hour",
            F.split("time_interval", "-")[0]
        )
        .withColumn(
            "traffic_hour",
            F.split("traffic_hour", ":")[0].cast("int")
        )
        .dropDuplicates([
            "segment_id",
            "traffic_date",
            "time_interval"
        ])
    )

    # Merge into the existing Silver table
    if DeltaTable.isDeltaTable(
        spark,
        "/Volumes/workspace/default/urbanpulse_bronze/_traffic_staging"
    ):
        pass

    # Use a unique staging table for this file
    (
        daily_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(staging_table)
    )

    target = DeltaTable.forName(spark, target_table)
    source = spark.table(staging_table)

    (
        target.alias("t")
        .merge(
            source.alias("s"),
            """
            t.segment_id = s.segment_id
            AND t.traffic_date = s.traffic_date
            AND t.time_interval = s.time_interval
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Completed: {file.name}")

    # Release references before the next file
    del raw_df, features, metadata, segments
    del date_ranges, time_sets, daily_df, source

    try:
        spark.catalog.clearCache()
    except Exception:
        # clearCache is not supported on serverless compute
        pass

print("\nAll remaining daily files processed!")
print("Final Silver records:", spark.table(target_table).count())


In [0]:
from pyspark.sql import functions as F

traffic_df = spark.table("workspace.default.silver_traffic")

traffic_df.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("traffic_date").alias("unique_dates"),
    F.countDistinct("segment_id").alias("unique_road_segments"),
    F.sum(F.col("traffic_date").isNull().cast("int")).alias("null_dates"),
    F.sum(F.col("segment_id").isNull().cast("int")).alias("null_segment_ids"),
    F.sum(F.col("probe_count").isNull().cast("int")).alias("null_probe_counts")
).show()

traffic_df.groupBy("traffic_date").count().orderBy("traffic_date").show(25)

In [0]:
probe_path = (
    "/Volumes/workspace/default/urbanpulse_bronze/"
    "traffic/probe_counts/geojson"
)

def find_probe_files(path):
    files = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(find_probe_files(item.path))
        elif item.name.lower().endswith((".json", ".geojson")):
            files.append(item)

    return files

probe_files = find_probe_files(probe_path)

print("Daily traffic files found:", len(probe_files))

for file in sorted(probe_files, key=lambda x: x.name):
    print(file.name, "|", round(file.size / (1024 * 1024), 2), "MB")

In [0]:
from pyspark.sql import functions as F
from functools import reduce

daily_traffic_dfs = []

for file in sorted(probe_files, key=lambda x: x.name):

    traffic_df = spark.read.json(file.path)

    features = traffic_df.select(
        F.explode("features").alias("feature")
    )

    date_ranges = (
        features
        .filter(F.col("feature.properties.dateRanges").isNotNull())
        .select(F.explode("feature.properties.dateRanges").alias("dr"))
        .select(
            F.col("dr.`@id`").alias("date_range_id"),
            F.to_date("dr.from").alias("traffic_date"),
            F.to_date("dr.to").alias("end_date")
        )
        .dropDuplicates()
    )

    time_sets = (
        features
        .filter(F.col("feature.properties.timeSets").isNotNull())
        .select(F.explode("feature.properties.timeSets").alias("ts"))
        .select(
            F.col("ts.`@id`").alias("time_set_id"),
            F.explode("ts.dayToTimeRanges").alias("day")
        )
        .select(
            "time_set_id",
            F.explode("day.timeRanges").alias("time_interval")
        )
        .dropDuplicates()
    )

    probe_summary = (
        features
        .filter(F.col("feature.properties.segmentProbeCounts").isNotNull())
        .select(
            F.col("feature.properties.segmentId").alias("segment_id"),
            F.col("feature.properties.streetName").alias("street_name"),
            F.col("feature.properties.frc").alias("frc"),
            F.col("feature.properties.distance").alias("distance"),
            F.col("feature.properties.speedLimit").alias("speed_limit"),
            F.col("feature.geometry.coordinates").alias("coordinates"),
            F.explode("feature.properties.segmentProbeCounts").alias("probe")
        )
        .select(
            "segment_id",
            "street_name",
            "frc",
            "distance",
            "speed_limit",
            "coordinates",
            F.col("probe.dateRange").alias("date_range_id"),
            F.col("probe.timeSet").alias("time_set_id"),
            F.col("probe.probeCount").alias("probe_count")
        )
    )

    enriched = (
        probe_summary
        .join(date_ranges, "date_range_id", "left")
        .join(time_sets, "time_set_id", "left")
    )

    daily_traffic_dfs.append(enriched)

traffic_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    daily_traffic_dfs
)

In [0]:
traffic_silver = (
    traffic_all
    .filter(F.col("segment_id").isNotNull())
    .filter(F.col("traffic_date").isNotNull())
    .filter(F.col("time_interval").isNotNull())
    .filter(F.col("probe_count").isNotNull())
    .filter(F.col("probe_count") >= 0)
    .filter(F.col("coordinates").isNotNull())
    .dropDuplicates([
        "segment_id",
        "traffic_date",
        "time_interval"
    ])
    .withColumn(
        "traffic_hour",
        F.split(F.col("time_interval"), "-")[0]
    )
    .withColumn(
        "traffic_hour",
        F.split(F.col("traffic_hour"), ":")[0].cast("integer")
    )
    .withColumn(
        "street_name",
        F.coalesce(F.trim(F.col("street_name")), F.lit("Unknown"))
    )
    .withColumn("processed_at", F.current_timestamp())
)

In [0]:
from pyspark.sql import functions as F
from functools import reduce

# Rebuild the per-file pipeline with the "multiline" JSON option.
# The read in the upstream cell is missing this option, so the pretty-printed
# 2024-08-11 GeoJSON yields only _corrupt_record and "features" cannot be
# resolved at write time (lazy evaluation).
daily_traffic_dfs = []

for file in sorted(probe_files, key=lambda x: x.name):

    traffic_df = (
        spark.read
        .option("multiline", "true")
        .json(file.path)
    )

    features = traffic_df.select(
        F.explode("features").alias("feature")
    )

    date_ranges = (
        features
        .filter(F.col("feature.properties.dateRanges").isNotNull())
        .select(F.explode("feature.properties.dateRanges").alias("dr"))
        .select(
            F.col("dr.`@id`").alias("date_range_id"),
            F.to_date("dr.from").alias("traffic_date"),
            F.to_date("dr.to").alias("end_date")
        )
        .dropDuplicates()
    )

    time_sets = (
        features
        .filter(F.col("feature.properties.timeSets").isNotNull())
        .select(F.explode("feature.properties.timeSets").alias("ts"))
        .select(
            F.col("ts.`@id`").alias("time_set_id"),
            F.explode("ts.dayToTimeRanges").alias("day")
        )
        .select(
            "time_set_id",
            F.explode("day.timeRanges").alias("time_interval")
        )
        .dropDuplicates()
    )

    probe_summary = (
        features
        .filter(F.col("feature.properties.segmentProbeCounts").isNotNull())
        .select(
            F.col("feature.properties.segmentId").alias("segment_id"),
            F.col("feature.properties.streetName").alias("street_name"),
            F.col("feature.properties.frc").alias("frc"),
            F.col("feature.properties.distance").alias("distance"),
            F.col("feature.properties.speedLimit").alias("speed_limit"),
            F.col("feature.geometry.coordinates").alias("coordinates"),
            F.explode("feature.properties.segmentProbeCounts").alias("probe")
        )
        .select(
            "segment_id",
            "street_name",
            "frc",
            "distance",
            "speed_limit",
            "coordinates",
            F.col("probe.dateRange").alias("date_range_id"),
            F.col("probe.timeSet").alias("time_set_id"),
            F.col("probe.probeCount").alias("probe_count")
        )
    )

    enriched = (
        probe_summary
        .join(date_ranges, "date_range_id", "left")
        .join(time_sets, "time_set_id", "left")
    )

    daily_traffic_dfs.append(enriched)

traffic_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    daily_traffic_dfs
)

# Reapply the cleaning transformations to the corrected data
traffic_silver = (
    traffic_all
    .filter(F.col("segment_id").isNotNull())
    .filter(F.col("traffic_date").isNotNull())
    .filter(F.col("time_interval").isNotNull())
    .filter(F.col("probe_count").isNotNull())
    .filter(F.col("probe_count") >= 0)
    .filter(F.col("coordinates").isNotNull())
    .dropDuplicates([
        "segment_id",
        "traffic_date",
        "time_interval"
    ])
    .withColumn(
        "traffic_hour",
        F.split(F.col("time_interval"), "-")[0]
    )
    .withColumn(
        "traffic_hour",
        F.split(F.col("traffic_hour"), ":")[0].cast("integer")
    )
    .withColumn(
        "street_name",
        F.coalesce(F.trim(F.col("street_name")), F.lit("Unknown"))
    )
    .withColumn("processed_at", F.current_timestamp())
)

(
    traffic_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_traffic")
)

In [0]:
traffic_hourly = (
    traffic_silver
    .groupBy(
        "traffic_date",
        "traffic_hour",
        "segment_id",
        "street_name",
        "frc"
    )
    .agg(
        F.sum("probe_count").alias("total_probe_count"),
        F.avg("speed_limit").alias("average_speed_limit"),
        F.avg("distance").alias("average_segment_distance")
    )
)

(
    traffic_hourly.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_traffic_hourly")
)